In [1]:
import pandas as pd
import numpy as np

In [2]:
errors = pd.read_csv('../Data/errors.csv')
failures = pd.read_csv('../Data/failures.csv')
maint = pd.read_csv('../Data/maint.csv')
telemetry = pd.read_csv('../Data/telemetry.csv')
machines = pd.read_csv('../Data/machines.csv')

In [3]:
for df in [errors, failures, maint, telemetry]:
    df['datetime'] = pd.to_datetime(df['datetime'])

In [4]:
for df in [errors, failures, maint, telemetry]:
    df.sort_values(['machineID','datetime'], inplace=True)

In [5]:
error_count = errors.groupby(['machineID','datetime']).size().reset_index(name='error_count')

In [6]:
df = df.merge(error_count, on=['machineID','datetime'], how='left')
df['error_count'].fillna(0, inplace=True)

C:\Users\user\AppData\Local\Temp\ipykernel_18996\4049509397.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['error_count'].fillna(0, inplace=True)


In [7]:
df['error_count'].value_counts()

error_count
0.0    872484
1.0      3342
2.0       245
3.0        29
Name: count, dtype: int64

In [8]:
failures['failure_flag'] = 1

In [9]:
df = df.merge(
    failures[['machineID','datetime','failure_flag']],
    on=['machineID','datetime'],
    how='left'
)

df['failure_flag'].fillna(0, inplace=True)

C:\Users\user\AppData\Local\Temp\ipykernel_18996\1603193733.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['failure_flag'].fillna(0, inplace=True)


In [10]:
df['failure_flag'].value_counts()

failure_flag
0.0    875381
1.0       761
Name: count, dtype: int64

In [11]:
df['target'] = df.groupby('machineID')['failure_flag'].shift(-1)
df.dropna(subset=['target'], inplace=True)

In [12]:
maint['maint_flag'] = 1
maint = maint.rename(columns={'datetime': 'last_maint_datetime'})

df = pd.merge_asof(
    df.sort_values('datetime'),
    maint.sort_values('last_maint_datetime'),
    left_on='datetime',
    right_on='last_maint_datetime',
    by='machineID',
    direction='backward'
)

df['days_since_maint'] = (df['datetime'] - df['last_maint_datetime']).dt.days
df['days_since_maint'].fillna(-1, inplace=True)  # -1 where no prior maint exists

C:\Users\user\AppData\Local\Temp\ipykernel_18996\4036559035.py:14: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['days_since_maint'].fillna(-1, inplace=True)  # -1 where no prior maint exists


In [13]:
df = df.merge(machines, on='machineID', how='left')

In [14]:
df['hour'] = df['datetime'].dt.hour
df['dayofweek'] = df['datetime'].dt.dayofweek
df['month'] = df['datetime'].dt.month
df['is_weekend'] = df['dayofweek'].isin([5,6]).astype(int)

In [15]:
cols = ['volt','rotate','pressure','vibration','error_count']

In [16]:
for col in cols:
    df[f'{col}_lag1'] = df.groupby('machineID')[col].shift(1)
    df[f'{col}_lag3'] = df.groupby('machineID')[col].shift(3)

In [17]:
for col in cols:
    df[f'{col}_mean_3'] = df.groupby('machineID')[col].transform(lambda x: x.rolling(3).mean())
    df[f'{col}_std_3']  = df.groupby('machineID')[col].transform(lambda x: x.rolling(3).std())
    df[f'{col}_max_3']  = df.groupby('machineID')[col].transform(lambda x: x.rolling(3).max())

In [18]:
for col in cols:
    df[f'{col}_diff'] = df[col] - df.groupby('machineID')[col].shift(1)

In [19]:
df['recent_maint'] = (df['days_since_maint'] < 7).astype(int)
df['log_days_since_maint'] = np.log1p(df['days_since_maint'])

In [20]:
df['error_rate'] = df['error_count'] / (df['error_count'].max() + 1)
df['error_trend'] = df.groupby('machineID')['error_count'].diff()

In [21]:
df['stress_index'] = df['vibration'] * df['pressure']
df['power_stress'] = df['volt'] * df['vibration']

In [22]:
df['machine_age'] = df['age']
df['model_encoded'] = df['model'].astype('category').cat.codes

In [23]:
df = df.dropna()

In [24]:
df.head()

,datetime,machineID,volt,rotate,pressure,vibration,error_count,failure_flag,target,last_maint_datetime,...,vibration_diff,error_count_diff,recent_maint,log_days_since_maint,error_rate,error_trend,stress_index,power_stress,machine_age,model_encoded
300,2015-01-01 09:00:00,77,192.250252,441.136665,102.321260,53.892747,0.0,0.0,0.0,2014-11-13 06:00:00,...,20.372756,0.0,0,3.912023,0.0,0.0,5514.373840,10360.894244,12,3
301,2015-01-01 09:00:00,10,178.128852,438.376292,105.870821,40.836263,0.0,0.0,0.0,2014-10-14 06:00:00,...,2.860985,0.0,0,4.382027,0.0,0.0,4323.368732,7274.116711,10,2
302,2015-01-01 09:00:00,8,171.170268,485.067147,97.887247,34.876035,0.0,0.0,0.0,2014-12-13 06:00:00,...,-3.524711,0.0,0,2.995732,0.0,0.0,3413.919045,5969.740224,16,2
303,2015-01-01 09:00:00,17,167.836488,483.978488,111.113911,66.352721,0.0,0.0,0.0,2015-01-01 06:00:00,...,18.565954,0.0,1,0.000000,0.0,0.0,7372.710388,11136.407766,14,0
304,2015-01-01 09:00:00,31,177.526310,486.244378,102.610134,50.725918,0.0,0.0,0.0,2014-09-29 06:00:00,...,0.459389,0.0,0,4.553877,0.0,0.0,5204.993271,9005.185142,11,0


In [25]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 875742 entries, 300 to 876041
Data columns (total 57 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   datetime              875742 non-null  datetime64[ns]
 1   machineID             875742 non-null  int64         
 2   volt                  875742 non-null  float64       
 3   rotate                875742 non-null  float64       
 4   pressure              875742 non-null  float64       
 5   vibration             875742 non-null  float64       
 6   error_count           875742 non-null  float64       
 7   failure_flag          875742 non-null  float64       
 8   target                875742 non-null  float64       
 9   last_maint_datetime   875742 non-null  datetime64[ns]
 10  comp                  875742 non-null  object        
 11  maint_flag            875742 non-null  int64         
 12  days_since_maint      875742 non-null  int64         
 13  mo

In [26]:
df.columns

Index(['datetime', 'machineID', 'volt', 'rotate', 'pressure', 'vibration',
       'error_count', 'failure_flag', 'target', 'last_maint_datetime', 'comp',
       'maint_flag', 'days_since_maint', 'model', 'age', 'hour', 'dayofweek',
       'month', 'is_weekend', 'volt_lag1', 'volt_lag3', 'rotate_lag1',
       'rotate_lag3', 'pressure_lag1', 'pressure_lag3', 'vibration_lag1',
       'vibration_lag3', 'error_count_lag1', 'error_count_lag3', 'volt_mean_3',
       'volt_std_3', 'volt_max_3', 'rotate_mean_3', 'rotate_std_3',
       'rotate_max_3', 'pressure_mean_3', 'pressure_std_3', 'pressure_max_3',
       'vibration_mean_3', 'vibration_std_3', 'vibration_max_3',
       'error_count_mean_3', 'error_count_std_3', 'error_count_max_3',
       'volt_diff', 'rotate_diff', 'pressure_diff', 'vibration_diff',
       'error_count_diff', 'recent_maint', 'log_days_since_maint',
       'error_rate', 'error_trend', 'stress_index', 'power_stress',
       'machine_age', 'model_encoded'],
      dtype='obj

In [27]:
df['target'].value_counts()

target
0.0    874981
1.0       761
Name: count, dtype: int64

In [29]:
df.to_csv('../Data/final_dataset.csv', index=False)